#Init

In [0]:
import pyspark.sql.functions as F

#Reading from Silver Table

In [0]:
silver_df = spark.read.table("adguard_telemetry.silver.adguard_clean")
display(silver_df)

#Business Logic

## Dim Client: Unique devices on the network

In [0]:
dim_client = silver_df.select("client","client_name").distinct().withColumn("client_sk",F.monotonically_increasing_id())
display(dim_client)

## Writing dim_client table into the Gold Layer

In [0]:
dim_client.write.mode("overwrite").saveAsTable("adguard_telemetry.gold.dim_client")

## Dim Domain: Unique websites requested

In [0]:
dim_domain = silver_df.select("query_domain").distinct().withColumn("domain_sk",F.monotonically_increasing_id())
display(dim_domain)

## Writing dim_domain into the Gold Layer

In [0]:
dim_domain.write.mode("overwrite").saveAsTable("adguard_telemetry.gold.dim_domain")

## Fact Queries: The core telemetry events

In [0]:
fact_queries = (silver_df
    .join(dim_client, on=["client","client_name"], how="left")
    .join(dim_domain, on="query_domain", how="left")
    .select(
        F.col("query_time"),
        F.col("client_sk"),
        F.col("domain_sk"),
        F.col("query_type"),
        F.col("reason"),
        F.col("elapsed_ms").alias("processing_time_ms"),
        F.col("cached")
    )
)
display(fact_queries)

## Writing fact_queries into the Gold Layer

In [0]:
fact_queries.write.mode("overwrite").saveAsTable("adguard_telemetry.gold.fact_queries")